# TP1 — Sentiment Classification on French Movie Reviews (Allociné)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session5/tp1.ipynb)

## Objective

In this TP we will:
1. Load and explore a **text dataset** (French movie reviews)
2. Clean and preprocess text data
3. Convert text to numerical features using **TF-IDF vectorization**
4. Train a **Logistic Regression** classifier on text
5. Interpret the model by examining learned coefficients

### Dataset: Allociné Movie Reviews

We use a subset of the **Allociné** dataset — 5,000 French movie reviews labeled as **positive** (1) or **negative** (0).

### New Concepts

| Concept | Description |
|---------|-------------|
| **TF-IDF** | Term Frequency–Inverse Document Frequency — weights words by how important they are in a document relative to the corpus |
| **Vectorization** | Converting text into a numerical matrix that models can use |
| **Stopwords** | Common words ("le", "de", "est") that carry little meaning |
| **n-grams** | Sequences of n consecutive words — bigrams capture phrases like "pas bon" |

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
import warnings
warnings.filterwarnings('ignore')

---
## 2. Data Loading & Exploration

We use pre-sampled CSVs from the Allociné dataset (5k train / 1k test). Each row has:
- `review` — the movie review text (French)
- `label` — 0 (negative) or 1 (positive)

### Exercise 1.1 — Load the data

Load the train and test CSVs and display the shape, columns, and first 5 rows.

*Hint:* `pd.read_csv(url)` can load directly from a URL.

In [ ]:
train_url = "https://raw.githubusercontent.com/racousin/L2Math/main/session5/allocine_train.csv"
test_url = "https://raw.githubusercontent.com/racousin/L2Math/main/session5/allocine_test.csv"

# YOUR CODE HERE

### Exercise 1.2 — Class balance

Check the distribution of positive vs negative reviews in the training set.

*Hint:* `df['label'].value_counts(normalize=True)`

In [ ]:
# YOUR CODE HERE

### Exercise 1.3 — Review length distribution

Compute the number of words per review and plot a histogram of review lengths, colored by label.

*Hint:* `df['n_words'] = df['review'].apply(lambda x: len(x.split()))`, then `sns.histplot(..., hue='label')`

In [ ]:
# YOUR CODE HERE

### Exercise 1.4 — Sample reviews

Display 3 positive and 3 negative reviews to get a feel for the data.

*Hint:* `df[df['label'] == 1]['review'].head(3)`

In [ ]:
# YOUR CODE HERE

---
## 3. Text Cleaning

Before vectorizing, we clean the text to reduce noise. Common preprocessing steps:
- **Lowercase** — so "Bon" and "bon" are the same token
- **Remove HTML tags** — leftover markup from web scraping
- **Remove URLs** — links carry no sentiment
- **Remove special characters and digits** — keep only letters and spaces

### Exercise 2.1 — Write a cleaning function

Write a function `clean_text(text)` that:
1. Converts to lowercase
2. Removes HTML tags (`<...>`)
3. Removes URLs
4. Removes digits
5. Removes special characters (keep only letters, accented characters, and spaces)

Apply it to the train and test review columns.

*Hint:*
```python
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)        # HTML tags
    text = re.sub(r'http\S+', ' ', text)         # URLs
    text = re.sub(r'\d+', ' ', text)              # digits
    text = re.sub(r'[^a-zàâäéèêëïîôùûüÿçœæ\s]', ' ', text)  # keep letters + accents
    text = re.sub(r'\s+', ' ', text).strip()      # multiple spaces
    return text
```

In [ ]:
# YOUR CODE HERE

### Exercise 2.2 — Before / after comparison

Show the original and cleaned version of 3 reviews side by side.

*Hint:* Print pairs of `df['review'].iloc[i]` and `df['review_clean'].iloc[i]`

In [ ]:
# YOUR CODE HERE

**Question:** Should we remove accents for French text? What about emojis? Numbers?

*YOUR ANSWER HERE*

---
## 4. Vectorization Experiments

**TF-IDF** (Term Frequency–Inverse Document Frequency) converts text into a numerical matrix:
- **TF**: how often a word appears in a document
- **IDF**: penalizes words that appear in many documents (common words get lower weight)
- **TF-IDF** = TF × IDF — words that are frequent in a document but rare across the corpus get the highest scores

We will experiment with different `TfidfVectorizer` configurations to see their impact.

### Exercise 3.1 — Build 4 TF-IDF configurations

Create 4 different `TfidfVectorizer` setups and fit-transform the training data:

| Config | Description |
|--------|-------------|
| **v1** | Default (unigrams only) |
| **v2** | With French stopwords |
| **v3** | Bigrams: `ngram_range=(1, 2)` |
| **v4** | Stopwords + bigrams + `max_features=20000` |

For each, print:
- Vocabulary size (`len(vectorizer.vocabulary_)`)
- Matrix shape

*Hint:*
```python
# French stopwords (minimal list)
french_stopwords = [
    'le', 'la', 'les', 'de', 'du', 'des', 'un', 'une',
    'et', 'est', 'en', 'que', 'qui', 'dans', 'ce', 'il',
    'ne', 'se', 'pas', 'plus', 'son', 'sur', 'au', 'avec',
    'tout', 'mais', 'par', 'pour', 'nous', 'vous', 'sa',
    'cette', 'ou', 'a', 'sont', 'je', 'ai', 'on', 'elle',
    'être', 'avoir', 'fait', 'comme', 'très', 'aussi',
    'même', 'bien', 'leurs', 'où', 'ces'
]

v1 = TfidfVectorizer()
v2 = TfidfVectorizer(stop_words=french_stopwords)
v3 = TfidfVectorizer(ngram_range=(1, 2))
v4 = TfidfVectorizer(stop_words=french_stopwords, ngram_range=(1, 2), max_features=20000)
```

In [ ]:
# YOUR CODE HERE

### Exercise 3.2 — Top-20 highest IDF terms

For each vectorizer, display the 20 terms with the **highest IDF** scores.

**Why:** High IDF means the term is rare across the corpus — these are the most discriminative words.

*Hint:*
```python
feature_names = vectorizer.get_feature_names_out()
idf_scores = vectorizer.idf_
top_20_idx = np.argsort(idf_scores)[-20:]
for idx in top_20_idx:
    print(f"{feature_names[idx]}: {idf_scores[idx]:.2f}")
```

In [ ]:
# YOUR CODE HERE

**Question:** Why does the vocabulary size explode when using bigrams? What are the implications for model training?

*YOUR ANSWER HERE*

---
## 5. Classification & Evaluation

Now we train a `LogisticRegression` on each of the 4 TF-IDF configurations and compare performance.

### Exercise 4.1 — Train and evaluate on all 4 configs

For each vectorizer config:
1. Transform train and test sets
2. Train a `LogisticRegression(max_iter=1000)`
3. Compute accuracy and F1-score on the test set

*Hint:*
```python
for name, vectorizer in configs.items():
    X_train_vec = vectorizer.fit_transform(train['review_clean'])
    X_test_vec = vectorizer.transform(test['review_clean'])
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train_vec, train['label'])
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(test['label'], y_pred)
    f1 = f1_score(test['label'], y_pred)
```

In [ ]:
# YOUR CODE HERE

### Exercise 4.2 — Confusion matrix for best config

Display the confusion matrix for the best-performing configuration.

*Hint:* `ConfusionMatrixDisplay.from_predictions(y_test, y_pred)`

In [ ]:
# YOUR CODE HERE

### Exercise 4.3 — Summary table

Create a summary `DataFrame` comparing all 4 configs: vocabulary size, accuracy, F1-score.

*Hint:* `pd.DataFrame(results)`

In [ ]:
# YOUR CODE HERE

**Question:** Which preprocessing choice has the biggest impact on performance? Why?

*YOUR ANSWER HERE*

---
## 6. Interpretation

One major advantage of Logistic Regression on TF-IDF: the model is **interpretable**. Each feature (word or bigram) has a coefficient — positive coefficients push toward class 1 (positive), negative toward class 0 (negative).

### Exercise 5.1 — Top positive and negative words

Using the best model, extract the top 15 words with the highest positive coefficients and top 15 with the most negative coefficients. Plot them as a horizontal barplot.

*Hint:*
```python
feature_names = best_vectorizer.get_feature_names_out()
coefs = best_model.coef_[0]
top_pos_idx = np.argsort(coefs)[-15:]
top_neg_idx = np.argsort(coefs)[:15]
```

In [ ]:
# YOUR CODE HERE

### Exercise 5.2 — Misclassified reviews

Find 2 misclassified reviews (1 false positive, 1 false negative). Read them and explain why the model might have failed.

**Look for:** irony, negation ("pas mauvais"), mixed sentiment, ambiguity.

*Hint:*
```python
errors = test[test['label'] != y_pred]
false_positives = errors[errors['label'] == 0]
false_negatives = errors[errors['label'] == 1]
```

In [ ]:
# YOUR CODE HERE

---
## 7. Bonus

### Exercise 6.1 — CountVectorizer vs TfidfVectorizer

Replace `TfidfVectorizer` with `CountVectorizer` (same config as your best TF-IDF) and compare.

**Question:** Does TF-IDF weighting help compared to raw counts?

In [ ]:
# YOUR CODE HERE

### Exercise 6.2 — SGDClassifier (linear SVM)

Replace `LogisticRegression` with `SGDClassifier(loss='hinge')` (equivalent to a linear SVM). Compare with Logistic Regression.

*Hint:* `SGDClassifier(loss='hinge', max_iter=1000, random_state=42)`

In [ ]:
# YOUR CODE HERE